# Grid Trading Bot, Backtested
The most popular "passive" bot on exchanges: pick a price range, the bot places limit orders on a
ladder of levels, buys one step lower, sells one step higher. Exchanges show a green "grid profit"
almost every time. Binance's own FAQ (frequently asked questions) says: Total Profit = Grid Profit +
Unrealized PnL (profit and loss). We start a new bot on the first day of every month, February 2021
to October 2026, on Bitcoin spot minute candles, and count both numbers. Education, not financial advice.

Our settings: 10,000 USDT (Tether, a dollar stablecoin), the range = the lowest and highest price of
the previous 30 days, 20 geometric grids, a 0.1% fee per order (the spot fee in Binance's own example),
one month per bot.

## 1. Helpers from episode q00
`binance_vision` — raw Binance data with SHA-256 hash checks (episode q00). No freqtrade this time: a grid is 20 limit orders resting in the book at once, and freqtrade holds one position per coin, so we write the order book ourselves.

In [ ]:
!curl -fsSLO https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/binance_vision.py
from binance_vision import download_all, months
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Bitcoin spot, every minute
Monthly archives of BTCUSDT spot 1-minute candles, each checked against Binance's SHA-256 hash (`download_all`, episode q00). From January 2025 spot timestamps are in microseconds, not milliseconds. Uses `download_all`, `months` (section 1).

In [ ]:
SPOT = "https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1m"
parts = download_all([f"{SPOT}/BTCUSDT-1m-{m}.zip"
                      for m in months("2021-01", "2026-10")])
m = pd.concat([p.iloc[:, :5] for p in parts], ignore_index=True)
m.columns = ["date", "open", "high", "low", "close"]
ms = np.where(m["date"] > 1e14, m["date"] // 1000, m["date"])  # us -> ms
m["date"] = pd.to_datetime(ms, unit="ms", utc=True)
m = m.drop_duplicates("date").set_index("date").sort_index()
print(f"{len(m):,} minutes")
print(m["close"].iloc[[0, -1]].to_string())

## 3. The levels
Geometric grid: $g_k = L \cdot (U/L)^{k/N}$, $k = 0 \dots N$ — L = lower price, U = upper price, N = grids; every step is the same percentage, $(U/L)^{1/N} - 1$. The range of a bot = the lowest and highest price of the 30 days before it starts. Uses `m` (section 2).

In [ ]:
def levels(low, high, n):
    return low * (high / low) ** (np.arange(n + 1) / n)   # same % per step

def month(start):
    before = m.loc[start - pd.Timedelta("30D"):start - pd.Timedelta("1min")]
    bars = m.loc[start:start + pd.DateOffset(months=1) - pd.Timedelta("1min")]
    return bars, before["low"].min(), before["high"].max()

april = pd.Timestamp("2021-04-01", tz="UTC")
bars, low, high = month(april)
g = levels(low, high, 20)
print(f"range {low:,.0f} to {high:,.0f} USDT, "
      f"step {(high / low) ** (1 / 20) - 1:.2%}")
print(np.round(g).astype(int))

## 4. The start
Every grid cell (from one level to the next) gets the same amount of Bitcoin, q. Cells above the starting price need a coin to sell, so the bot buys it at market right away (Binance's FAQ: "automatically place market orders"); cells below wait with cash for their buy order. Uses `g`, `bars` (3).

In [ ]:
FEE = 0.001                                       # 0.1% of every order

def start_bot(g, p0, money=10_000, fee=FEE):
    hold = g[:-1] >= p0                    # cells above the price: coin now
    cost = np.where(hold, p0, g[:-1]) * (1 + fee)    # price paid per coin
    q = money / cost.sum()                 # the same coins in every cell
    return hold, cost, q, money - q * cost[hold].sum()

p0 = bars["open"].iloc[0]
hold, cost, q, cash = start_bot(g, p0)
print(f"start price {p0:,.0f}: {hold.sum()} of 20 cells bought at once")
print(f"{q:.6f} BTC per cell; {cash:,.0f} USDT left for the buys below")

## 5. Inside a minute
A candle gives open, high, low and close, not the order in between. Our assumption: a rising minute went open → low → high → close, a falling one open → high → low → close; section 13 tries the opposite order.

In [ ]:
def path(o, h, l, c, flip=False):
    up = (c >= o) != flip
    return (o, l, h, c) if up else (o, h, l, c)

## 6. The bot, minute by minute
Price falling through a level → that cell's buy fills at the level, if the cash is there (an exchange never lends to a grid bot). Price rising through the level above a held cell → it sells there: grid profit = sell − buy − both fees, one matched pair. At the end, the coins still held are valued at the last price: unrealized PnL (profit and loss). The assert checks that the cash never goes below zero, and Binance's identity, Total Profit = Grid Profit + Unrealized PnL. Uses `levels` (3), `start_bot`, `FEE` (4), `path` (5).

In [ ]:
def run_grid(bars, low, high, n=20, money=10_000, fee=FEE, flip=False):
    g = levels(low, high, n)
    hold, cost, q, cash = start_bot(g, bars["open"].iloc[0], money, fee)
    grid, days = 0.0, {}
    eod = (bars.index.hour == 23) & (bars.index.minute == 59)
    o, h, l, c = (bars[k].to_numpy() for k in ("open", "high", "low", "close"))
    for i in range(len(o)):
        p = path(o[i], h[i], l[i], c[i], flip)
        for a, b in zip(p[:-1], p[1:]):
            if b < a:                      # falling: buys at levels in [b, a)
                k = np.nonzero(~hold & (g[:-1] >= b) & (g[:-1] < a))[0][::-1]
                k = k[np.cumsum(q * g[k] * (1 + fee)) <= cash]  # cash only
                cost[k] = g[k] * (1 + fee)
                cash -= q * cost[k].sum(); hold[k] = True
            elif b > a:                    # rising: sell one level higher
                k = np.nonzero(hold & (g[1:] <= b) & (g[1:] > a))[0]
                sold = q * g[k + 1] * (1 - fee)
                cash += sold.sum(); hold[k] = False
                grid += (sold - q * cost[k]).sum()  # matched pairs
        if eod[i]:
            days[bars.index[i].normalize()] = cash + q * hold.sum() * c[i]
    value = cash + q * hold.sum() * c[-1]
    unrealized = (q * (c[-1] - cost[hold])).sum()
    assert cash > -1e-6 and abs(value - money - grid - unrealized) < 1e-6
    hold_btc = money * (c[-1] / o[0] - 1)
    return dict(grid=grid, unrealized=unrealized, total=value - money,
                hold=hold_btc, days=pd.Series(days))

## 7. April 2021: a sideways month
Uses `month` (3), `run_grid` (6), `april` (3).

In [ ]:
r = run_grid(*month(april))
for k in ("grid", "unrealized", "total", "hold"):
    print(f"{k:>10}: {r[k]:>+8,.0f} USDT")

## 8. Two months out of the range
June 2022: the price fell below the range. February 2024: it rose above. Uses `month` (3), `run_grid` (6).

In [ ]:
for start in ("2022-06-01", "2024-02-01"):
    t = pd.Timestamp(start, tz="UTC")
    bars, low, high = month(t)
    r = run_grid(bars, low, high)
    print(start, f"range {low:,.0f}-{high:,.0f}, "
          f"price {bars.open.iloc[0]:,.0f} -> {bars.close.iloc[-1]:,.0f}")
    print(f"  grid {r['grid']:+,.0f}  unrealized {r['unrealized']:+,.0f}"
          f"  total {r['total']:+,.0f}  hold {r['hold']:+,.0f}")

## 9. A new bot every month, 69 months
February 2021 to October 2026, each bot with 10,000 USDT (Tether, a dollar stablecoin). "Reinvested" chains the months: every bot starts with what the last one ended with. Uses `month` (3), `run_grid` (6).

In [ ]:
def every_month(**kw):
    rows, days = [], []
    for t in pd.date_range("2021-02-01", "2026-10-01", freq="MS", tz="UTC"):
        r = run_grid(*month(t), **kw)
        rows.append({"start": t.date(), **{k: r[k] for k in
                     ("grid", "unrealized", "total", "hold")}})
        days.append(r["days"] / 10_000)            # account per 1 USDT
    return pd.DataFrame(rows).set_index("start"), days

bots, bot_days = every_month()
print(bots.round(0).head(3).to_string())
print("grid profit > 0:", (bots["grid"] > 0).sum(), "of", len(bots),
      f"  sum {bots['grid'].sum():+,.0f}")
print("total > 0:      ", (bots["total"] > 0).sum(), "of", len(bots),
      f"  sum {bots['total'].sum():+,.0f}")
print("total > hold:   ", (bots.total > bots.hold).sum(), "of", len(bots))
print(f"reinvested: x{(1 + bots['total'] / 10_000).prod():.3f}"
      f"   hold Bitcoin: x{(1 + bots['hold'] / 10_000).prod():.3f}")

## 10. More grids, fewer grids, no fee
The same 69 months with 10, 50, 100 and 200 grids, and 20 grids without the fee. A step must be wider than two fees, 0.2%, or every "win" loses money. Uses `every_month` (9), `bots` (9).

In [ ]:
variants = {"20 grids": bots}
for name, kw in [("10 grids", dict(n=10)), ("50 grids", dict(n=50)),
                 ("100 grids", dict(n=100)), ("200 grids", dict(n=200)),
                 ("20 grids, no fee", dict(fee=0.0))]:
    variants[name], _ = every_month(**kw)
for name, b in variants.items():
    print(f"{name:>16}: grid > 0 in {(b['grid'] > 0).sum():>2} of {len(b)},"
          f" total {b['total'].sum():>+8,.0f},"
          f" reinvested x{(1 + b['total'] / 10_000).prod():.3f}")

## 11. The account, month after month
Every bot's account at the end of each day, chained (reinvested), against holding Bitcoin. Uses `bot_days` (9), `m` (2).

In [ ]:
grid_curve = []
level = 10_000.0
for d in bot_days:                     # one month: account per 1 USDT
    grid_curve.append(level * d)
    level *= d.iloc[-1]
grid_curve = pd.concat(grid_curve)
btc = m["close"].resample("1D").last().loc["2021-02-01":"2026-10-31"]
hold_curve = 10_000 * btc / m.loc["2021-02-01", "open"].iloc[0]
for name, curve in (("grid bot", grid_curve), ("hold Bitcoin", hold_curve)):
    fall = (1 - curve / curve.cummax()).max()
    print(f"{name:>12}: final {curve.iloc[-1]:>8,.0f} max drawdown {fall:.0%}")
plt.figure(figsize=(11, 5))
plt.plot(grid_curve.index, grid_curve, label="grid bot, reinvested")
plt.plot(hold_curve.index, hold_curve, label="hold Bitcoin")
plt.yscale("log"); plt.legend(); plt.title("10,000 USDT, 2021 to 2026")
plt.show()

## 12. The Sharpe ratio: return per unit of wobble
The same four lines as episode q03: Sharpe ratio $S = \dfrac{\text{mean of the daily returns}}{\text{their standard deviation}} \times \sqrt{365}$; risk-free rate 0: idle USDT (Tether, a dollar stablecoin) earns nothing here. Uses `grid_curve`, `hold_curve` (11).

In [ ]:
def sharpe(curve):
    day = curve.resample("1D").last()           # the account once a day
    r = day.pct_change().dropna()               # daily returns
    return r.mean() / r.std() * 365 ** 0.5      # a year = 365 crypto days

for name, curve in (("grid bot", grid_curve), ("hold Bitcoin", hold_curve)):
    r = curve.resample("1D").last().pct_change().dropna()
    print(f"{name:>12}: mean {r.mean():.3%}  std {r.std():.2%}"
          f"  Sharpe {sharpe(curve):.2f}")

## 13. The other order inside a minute
Section 5 assumed a rising minute dipped first. The opposite order tests how much that guess matters. Uses `every_month` (9), `bots` (9).

In [ ]:
flipped, _ = every_month(flip=True)
for name, b in (("as assumed", bots), ("opposite order", flipped)):
    print(f"{name:>14}: grid {b['grid'].sum():>+8,.0f},"
          f" total {b['total'].sum():>+7,.0f},"
          f" reinvested x{(1 + b['total'] / 10_000).prod():.3f}")

## 14. Everything side by side
A legend explains every row. Uses `bots` (9), `variants` (10), `flipped` (13), `grid_curve`, `hold_curve` (11), `sharpe` (12).

In [ ]:
runs = {**variants, "20, opposite order": flipped}
def summary(b):                         # one row of the table
    return {"months": f"{len(b)}", "grid > 0": f"{(b['grid'] > 0).sum()}",
            "total > 0": f"{(b['total'] > 0).sum()}",
            "grid sum": f"{b['grid'].sum():+,.0f}",
            "total sum": f"{b['total'].sum():+,.0f}",
            "reinvest": f"x{(1 + b['total'] / 10_000).prod():.2f}"}
table = pd.DataFrame({name: summary(b) for name, b in runs.items()}).T
table.loc["hold Bitcoin"] = [f"{len(bots)}", "", f"{(bots['hold'] > 0).sum()}",
                             "", f"{bots['hold'].sum():+,.0f}",
                             f"x{(1 + bots['hold'] / 10_000).prod():.2f}"]
print(table.to_string())
print(f"Sharpe: grid bot {sharpe(grid_curve):.2f}, "
      f"hold Bitcoin {sharpe(hold_curve):.2f}")
print("money in USDT (Tether, a dollar stablecoin), each bot from 10,000")
print("grid > 0, total > 0: months in profit; sums over the 69 bots")
print("grid = grid profit: matched buy-sell pairs, fees deducted")
print("total = grid profit + unrealized PnL (profit and loss)")
print("reinvest = every month starts with what the last one ended with")
print("Sharpe = mean daily return / its standard deviation x sqrt(365);")
print("  above 1 is the usual target")